# Uranium-associated prospectivity modelling with nested Optuna and OOF TreeSHAP

This notebook is the thin orchestration layer for analysis revision `task_a_source_connected_optuna_shap_v7_3_convergence_audited`. Run the cells in order. Optuna studies are resumed only when every stored protocol attribute matches the current frozen protocol.

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
PACKAGE_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
CONFIG_PATH = PACKAGE_ROOT / 'config' / 'prospectivity_config.json'
SOURCE_DIR = PACKAGE_ROOT / 'src'
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

for module_name in [name for name in list(sys.modules) if name.startswith('task_a_')]:
    del sys.modules[module_name]

required = ['numpy', 'pandas', 'scipy', 'sklearn', 'openpyxl', 'matplotlib', 'optuna', 'xgboost', 'shap']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(
        'Missing packages in the active Jupyter kernel: ' + ', '.join(missing) +
        '. Create the environment described in environment.yml, restart the kernel, and run again.'
    )
print(f'Package root: {PACKAGE_ROOT}')
print(f'Configuration: {CONFIG_PATH}')

In [ ]:
from export_utils import load_config, package_environment, save_json, validate_runtime_environment
from data_pipeline import run_data_pipeline
from nested_tuning import run_repeated_nested_evaluation
from shap_analysis import run_selected_model_oof_shap
from mixed_group_challenge import run_mixed_group_challenge
from export_utils import run_read_only_exports

config, package_root, output_paths = load_config(CONFIG_PATH)
runtime_validation = validate_runtime_environment(CONFIG_PATH)
environment = package_environment()
save_json(output_paths['audit'] / 'runtime_environment.json', environment)
{'runtime_validation': runtime_validation, 'environment': environment}

## 1. Input audit, source-connected blocks, and frozen cohorts

The code validates rather than reconstructs labels. Reference-connected blocks are constructed on all traceable records before stable and mixed-label cohorts are separated.

In [ ]:
data_summary = run_data_pipeline(CONFIG_PATH)
data_summary

In [ ]:
import pandas as pd
pd.read_csv(output_paths['audit'] / 'cohort_flow.csv')

## 2. Repeated nested grouped OOF evaluation

Four algorithms receive the same Optuna budget inside each outer-training partition. The outer validation blocks are never read by Optuna or threshold selection. This is the computationally intensive stage.

In [ ]:
model_decision = run_repeated_nested_evaluation(CONFIG_PATH)
model_decision

In [ ]:
protocol_contract = json.loads((output_paths['audit'] / 'analysis_protocol_contract.json').read_text(encoding='utf-8'))
aggregation_protocol = json.loads((output_paths['audit'] / 'aggregation_protocol.json').read_text(encoding='utf-8'))
{'analysis_revision': protocol_contract['analysis_revision'],
 'analysis_protocol_hash': protocol_contract['analysis_protocol_hash'],
 'primary_group_aggregation': aggregation_protocol['primary_group_aggregation']}

In [ ]:
pd.read_csv(output_paths['results'] / 'model_performance_mean_ci.csv')

## 3. Selected-model paired OOF TreeSHAP

TreeSHAP is calculated only when the objectively selected model is RF or XGBoost. A non-tree winner is recorded as not applicable; no substitute model is introduced.

In [ ]:
shap_status = run_selected_model_oof_shap(CONFIG_PATH)
shap_status

## 4. Mixed-label challenge cohort

The selected algorithm's outer-training models predict the mixed-label records without using their labels. No binary performance metric is calculated for this challenge cohort.

In [ ]:
challenge_status = run_mixed_group_challenge(CONFIG_PATH)
challenge_status

## 5. Read-only figures, report, and output manifest

Figures read frozen OOF outputs and cannot alter tuning, model selection, thresholds, or SHAP gates.

In [ ]:
export_status = run_read_only_exports(CONFIG_PATH)
export_status

In [ ]:
integrity = json.loads((output_paths['audit'] / 'preflight_and_integrity_checks.json').read_text(encoding='utf-8'))
if not integrity.get('active_run_all_fatal_checks_passed', False):
    raise RuntimeError('One or more fatal integrity checks failed for the active run.')
trial_budget = pd.read_csv(output_paths['optuna'] / 'trial_budget_audit.csv')
expected_rows = len(config['validation']['models']) * config['validation']['outer_repeats'] * config['validation']['outer_folds']
if len(trial_budget) != expected_rows or not trial_budget['passed'].all():
    raise RuntimeError('The exact Optuna trial-budget acceptance criterion was not met.')
decision = json.loads((output_paths['results'] / 'model_selection_decision.json').read_text(encoding='utf-8'))
if decision['selected_model_is_tree_based']:
    gate = json.loads((output_paths['shap'] / 'global_attribution_reliability_gate.json').read_text(encoding='utf-8'))
    bridge_contract = json.loads((output_paths['bridge'] / 'taskA_bridge_contract.json').read_text(encoding='utf-8'))
    bridge = pd.read_csv(output_paths['bridge'] / 'taskA_oof_bridge_one_row_per_Record_ID.csv')
    if bridge['Record ID'].duplicated().any():
        raise RuntimeError('The coupling bridge is not one row per Record ID.')
else:
    gate = {'global_gate_passed': None}
    bridge_contract = json.loads((output_paths['bridge'] / 'taskA_bridge_contract.json').read_text(encoding='utf-8'))
required_challenge = [
    output_paths['challenge'] / 'mixed_group_summary_all_models.csv',
    output_paths['challenge'] / 'mixed_group_summary_nonoverlap_only.csv',
    output_paths['challenge'] / 'mixed_group_overlap_sensitivity.csv',
]
if challenge_status.get('status') != 'not_applicable_no_mixed_groups' and not all(path.exists() for path in required_challenge):
    raise RuntimeError('One or more mixed-group overlap-sensitivity outputs are missing.')
print('Uranium prospectivity model completed and passed the executable acceptance checks.')
print({'selected_model': decision['selected_model'],
       'global_attribution_gate_passed': gate['global_gate_passed'],
       'bridge_status': bridge_contract['bridge_status']})
print(output_paths['logs'] / 'uranium_prospectivity_model_report_CN.md')